# Natural Language Processing with Sequence Models

Если probabilistic NLP пытался моделировать текст как вероятностную последовательность через **n-grams** и **HMM**, то sequence models делают следующий шаг:

**они учатся представлять последовательность как динамическое состояние, которое обновляется при чтении каждого нового токена.**

Это был критический поворот в развитии NLP. Вместо короткого фиксированного контекста, как в trigram model, sequence models пытаются нести информацию через всю последовательность. В субтитрах курса это формулируется прямо: главное преимущество RNN — они распространяют информацию по последовательности, а вычислительные блоки используют общие параметры на каждом шаге; это позволяет учитывать зависимости, которые n-gram language models часто не способны захватить.

---

# 1. Место раздела в общей карте знаний NLP

```text
NLP
├── Text processing and classical models
│   ├── preprocessing
│   ├── BoW / TF-IDF
│   ├── Logistic Regression
│   └── Naive Bayes
├── Probabilistic sequence models
│   ├── n-gram language models
│   ├── smoothing
│   ├── Markov chains
│   └── HMM
├── Neural sequence models
│   ├── neural language models
│   ├── recurrent neural networks
│   ├── LSTM
│   ├── GRU
│   ├── bidirectional RNNs
│   ├── sequence tagging
│   ├── encoder-decoder / seq2seq
│   ├── decoding strategies
│   └── beam search
└── Attention and Transformers
    ├── attention
    ├── self-attention
    └── transformer architectures
```

## Что здесь фундаментально

Фундаментальные темы раздела:

- embeddings как вход в последовательностную модель;
- recurrent computation;
- hidden state;
- parameter sharing across time;
- backpropagation through time;
- vanishing/exploding gradients.

## Что здесь производно

Производные темы:

- RNN, LSTM, GRU;
- bidirectional models;
- seq2seq;
- sequence tagging;
- beam search;
- encoder-decoder generation.

Этот раздел особенно важен как мост между классическим NLP и attention models. Он показывает, почему вообще понадобились attention и transformers: **RNN сделали контекст длиннее, но не решили его идеально**.

---

# 2. Онтология ключевых терминов раздела

### Neural Language Model
- **тип:** neural probabilistic model
- **определение:** модель, предсказывающая следующее слово с помощью обучаемых векторных представлений и нейросетевых параметров.
- **связи:** embeddings, RNN, seq2seq.
- **роль:** переход от count-based LM к learned representations.

### Recurrent Neural Network (RNN)
- **тип:** neural sequence model
- **определение:** сеть, которая обновляет скрытое состояние при чтении очередного токена.
- **связи:** hidden state, BPTT, vanishing gradients, LSTM, GRU.
- **роль:** базовая архитектура sequence modeling.

### Hidden State
- **тип:** latent representation
- **определение:** вектор памяти, который суммирует информацию о префиксе последовательности.
- **связи:** RNN, LSTM, GRU, seq2seq.
- **роль:** внутренний контекст модели.

### Backpropagation Through Time (BPTT)
- **тип:** optimization procedure
- **определение:** обучение рекуррентной сети градиентным распространением ошибки через временные шаги.
- **связи:** gradient descent, vanishing gradient, exploding gradient.

### LSTM
- **тип:** gated recurrent architecture
- **определение:** разновидность RNN с ячейкой памяти и воротами, предназначенная для лучшего хранения долгосрочных зависимостей.
- **связи:** forget gate, input gate, output gate, seq2seq.

### GRU
- **тип:** gated recurrent architecture
- **определение:** упрощённый вариант LSTM с меньшим числом ворот.
- **связи:** update gate, reset gate, efficient recurrent modeling.

### Bidirectional RNN
- **тип:** sequence encoder
- **определение:** RNN, читающая последовательность слева направо и справа налево.
- **связи:** sequence tagging, context from both sides.

### Seq2Seq
- **тип:** encoder-decoder architecture
- **определение:** модель, отображающая одну последовательность в другую.
- **связи:** machine translation, LSTM encoder, decoder, attention.

### Beam Search
- **тип:** decoding algorithm
- **определение:** метод поиска наиболее вероятных выходных последовательностей, сохраняющий несколько лучших гипотез на каждом шаге.
- **связи:** seq2seq, language generation, translation.

### Sequence Tagging
- **тип:** NLP task family
- **определение:** задача назначения метки каждому токену входной последовательности.
- **связи:** POS tagging, NER, BiLSTM, CRF.

---

# 3. Почему понадобились sequence models

## 3.1. Интуитивная идея

Классические probabilistic models часто видят слишком короткий контекст. Trigram знает только два предыдущих слова. Это полезно, но часто недостаточно.

Курс показывает типичный пример: в предложении вида  
*I called her, but she did not blank*  
trigram может предпочесть слово, которое часто идёт после *did not*, но не учитывает, что ранее было *I called her*. RNN, напротив, переносит информацию от начала к концу и поэтому может предсказать более осмысленное продолжение, например *answer*.

То есть главная мотивация sequence models:

- учитывать более длинный контекст;
- параметризовать зависимости не через таблицы частот, а через обучаемые веса;
- строить обобщающие представления последовательности.

## 3.2. Формальный переход от n-gram к neural sequence model

В n-gram LM:

$$
P(w_t \mid w_{1:t-1}) \approx P(w_t \mid w_{t-n+1:t-1})
$$

В sequence model хотим приблизить:

$$
P(w_t \mid w_{1:t-1}) \approx P(w_t \mid h_{t-1})
$$

где $h_{t-1}$ — скрытое состояние, сжимающее всю предыдущую историю.

---

# 4. Neural Language Models

## 4.1. Интуитивная идея

Neural language model — это уже не просто таблица условных вероятностей n-gram. Здесь слова кодируются векторами, а вероятность следующего токена вычисляется через нейросетевое преобразование.

Главная идея:

- слова получают dense representations;
- контекст кодируется непрерывным вектором;
- вероятности выходов рождаются из learned parameters.

Это качественный шаг вперёд по сравнению с count-based models.

## 4.2. Формальная постановка

Пусть последовательность токенов:

$$
x_1, x_2, \dots, x_T
$$

Каждый токен сначала переходит в embedding:

$$
e_t = E[x_t]
$$

Далее модель поддерживает скрытое состояние $h_t$, а вероятность следующего слова:

$$
P(x_{t+1}\mid x_{1:t}) = \text{softmax}(Wh_t + b)
$$

## 4.3. Функция потерь

Для language modeling стандартная loss:

$$
\mathcal{L} = - \sum_{t=1}^{T-1} \log P(x_{t+1} \mid x_{1:t})
$$

Это cross-entropy по следующему токену.

## 4.4. Ограничения ранних neural LM

Хотя neural LM уже лучше n-grams, ранние модели с фиксированным окном всё ещё ограничены конечным контекстом. Именно это и приводит к рекуррентным сетям.

---

# 5. Recurrent Neural Networks (RNN)

## 5.1. Интуитивная идея

RNN читает последовательность токен за токеном. На каждом шаге она:

- получает текущий токен;
- получает предыдущее скрытое состояние;
- обновляет память;
- при необходимости делает предсказание.

Курс формулирует это очень наглядно: информация от начала последовательности переносится к концу, а на каждом шаге повторяется один и тот же вычислительный блок с одними и теми же параметрами.

## 5.2. Формальная постановка

Пусть $x_t$ — вход на шаге $t$, обычно embedding слова.

Тогда простая RNN задаётся:

$$
h_t = \phi(W_x x_t + W_h h_{t-1} + b_h)
$$

$$
o_t = W_y h_t + b_y
$$

$$
\hat y_t = g(o_t)
$$

где:

- $h_t$ — hidden state;
- $W_x$ — веса входа;
- $W_h$ — рекуррентные веса;
- $W_y$ — выходные веса;
- $\phi$ — нелинейность, например $\tanh$;
- $g$ — softmax или sigmoid, в зависимости от задачи.

## 5.3. Почему это важно

Вместо отдельных параметров для каждого положения в последовательности мы используем **общие параметры** на всех шагах:

$$
W_x, W_h, W_y
$$

Это даёт:

- масштабируемость;
- инвариантность к длине последовательности;
- возможность обрабатывать произвольную длину входа.

Курс прямо подчёркивает, что именно из-за повторного применения одного и того же блока сеть называется recurrent.

## 5.4. Алгоритм

1. Инициализировать $h_0$.
2. Для каждого токена $x_t$:
   - получить embedding;
   - вычислить $h_t$;
   - при необходимости вычислить $\hat y_t$.
3. Агрегировать loss.
4. Обновить параметры через BPTT.

## 5.5. Псевдокод

```text
h = h0
for t in 1..T:
    x_t = embedding(token_t)
    h = tanh(Wx @ x_t + Wh @ h + b)
    y_t = softmax(Wy @ h + by)   # если нужен output на каждом шаге
```

## 5.6. Примеры архитектур по типу input/output

Курс отдельно разбирает разные типы RNN-задач: one-to-one, one-to-many, many-to-one, many-to-many. Для NLP особенно важны:  
- **many-to-one** — sentiment analysis;  
- **many-to-many** — machine translation, sequence tagging;  
- **one-to-many** — caption generation.

### Many-to-one
$$
x_1,\dots,x_T \rightarrow y
$$

Пример: sentiment classification.

### Many-to-many
$$
x_1,\dots,x_T \rightarrow y_1,\dots,y_{T'}
$$

Пример: machine translation.

### Token-level many-to-many
$$
x_1,\dots,x_T \rightarrow y_1,\dots,y_T
$$

Пример: POS tagging, NER.

## 5.7. Ограничения простой RNN

Главная проблема — градиентные трудности на длинных последовательностях.

При backpropagation recurrent Jacobians многократно перемножаются. Это приводит к:

- **vanishing gradients**;
- **exploding gradients**.

Из-за этого простая RNN плохо удерживает долгосрочные зависимости.

---

# 6. Backpropagation Through Time (BPTT)

## 6.1. Интуитивная идея

RNN применяет один и тот же блок на каждом временном шаге. При обучении нужно распространить ошибку назад через всю развёрнутую во времени сеть.

Это и есть BPTT.

## 6.2. Формальная постановка

Если суммарная функция потерь:

$$
\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t
$$

то градиент по рекуррентным параметрам зависит от цепочки производных:

$$
\frac{\partial \mathcal{L}}{\partial W_h}
=
\sum_t \frac{\partial \mathcal{L}}{\partial h_t}
\frac{\partial h_t}{\partial W_h}
$$

Но $\frac{\partial \mathcal{L}}{\partial h_t}$ включает вклад всех будущих шагов, что делает обучение нестабильным.

## 6.3. Vanishing / Exploding Gradients

Если нормы матриц/производных меньше 1, то произведение экспоненциально затухает.  
Если больше 1 — взрывается.

Именно поэтому курс подчёркивает, что в seq2seq обычно используют LSTM и GRU, чтобы смягчить проблемы vanishing and exploding gradients.

## 6.4. Практические способы борьбы

- gradient clipping;
- LSTM / GRU;
- careful initialization;
- layer normalization;
- truncated BPTT.

---

# 7. LSTM

## 7.1. Интуитивная идея

LSTM решает проблему короткой памяти в обычной RNN. Вместо одного скрытого состояния она хранит отдельную **ячейку памяти** $c_t$, а специальные ворота управляют:

- что забыть;
- что записать;
- что выдать наружу.

Так сеть может удерживать релевантную информацию намного дольше.

В курсе LSTM фигурирует как основной строительный блок seq2seq-моделей именно потому, что plain RNN страдает от gradient problems.

## 7.2. Формальная постановка

Для LSTM:

### Forget gate
$$
f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)
$$

### Input gate
$$
i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)
$$

### Candidate memory
$$
\tilde c_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)
$$

### Cell update
$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t
$$

### Output gate
$$
o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)
$$

### Hidden state
$$
h_t = o_t \odot \tanh(c_t)
$$

## 7.3. Интерпретация ворот

- **forget gate** решает, какую часть старой памяти оставить;
- **input gate** контролирует, что записать;
- **output gate** контролирует, что показать на выходе.

## 7.4. Псевдокод

```text
for each time step t:
    f_t = sigmoid(Wf x_t + Uf h_{t-1} + bf)
    i_t = sigmoid(Wi x_t + Ui h_{t-1} + bi)
    c_hat = tanh(Wc x_t + Uc h_{t-1} + bc)
    c_t = f_t * c_{t-1} + i_t * c_hat
    o_t = sigmoid(Wo x_t + Uo h_{t-1} + bo)
    h_t = o_t * tanh(c_t)
```

## 7.5. Применение в NLP

- neural language modeling;
- machine translation;
- sequence tagging;
- text generation;
- dialogue modeling.

## 7.6. Ограничения

LSTM существенно лучше простой RNN, но:

- обучение всё ещё последовательное по времени;
- длинные зависимости всё равно не идеальны;
- трудно параллелить;
- фиксированное bottleneck-state для seq2seq остаётся проблемой.

---

# 8. GRU

## 8.1. Интуитивная идея

GRU — более компактный родственник LSTM. Он пытается сохранить преимущества gated memory, но сделать архитектуру проще.

## 8.2. Формальная постановка

### Update gate
$$
z_t = \sigma(W_z x_t + U_z h_{t-1})
$$

### Reset gate
$$
r_t = \sigma(W_r x_t + U_r h_{t-1})
$$

### Candidate hidden state
$$
\tilde h_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}))
$$

### Final hidden state
$$
h_t = (1-z_t)\odot h_{t-1} + z_t \odot \tilde h_t
$$

## 8.3. Интуиция

- **reset gate** управляет тем, насколько учитывать прошлое при формировании нового кандидата;
- **update gate** управляет тем, сколько старой памяти сохранить.

## 8.4. Практическая роль

GRU часто:

- быстрее;
- легче;
- иногда показывает качество не хуже LSTM.

Поэтому курс и упоминает LSTM и GRU вместе как типичные решения для seq2seq и recurrent NLP.

---

# 9. Bidirectional RNN

## 9.1. Интуитивная идея

Обычная RNN кодирует только левый контекст:

$$
x_1,\dots,x_t
$$

Но во многих NLP-задачах для токена важны и левые, и правые соседи. Например, в sequence tagging слово лучше интерпретировать, видя всю фразу.

Тогда используют две сети:

- forward RNN;
- backward RNN.

## 9.2. Формальная постановка

$$
\overrightarrow{h_t} = \text{RNN}_f(x_t, \overrightarrow{h_{t-1}})
$$

$$
\overleftarrow{h_t} = \text{RNN}_b(x_t, \overleftarrow{h_{t+1}})
$$

Итоговое представление:

$$
h_t = [\overrightarrow{h_t}; \overleftarrow{h_t}]
$$

## 9.3. Преимущество

BiRNN использует **контекст с двух сторон**, что особенно полезно в:

- POS tagging;
- NER;
- chunking;
- slot filling.

---

# 10. Sequence Tagging

## 10.1. Интуитивная идея

В sequence tagging нужно выдать метку для каждого токена:

- POS tag;
- named entity label;
- chunk tag.

Курс ещё в probabilistic block подчёркивал важность POS tagging и его роль в NLP. Sequence models позволяют решать такие задачи уже не через HMM, а через learned contextual encoders.

## 10.2. Формальная постановка

Пусть вход:

$$
x_1,\dots,x_T
$$

Тогда нужно предсказать:

$$
y_1,\dots,y_T
$$

Обычно:

$$
h_t = \text{BiLSTM}(x_{1:T}, t)
$$

$$
\hat y_t = \text{softmax}(W h_t + b)
$$

## 10.3. BiLSTM-CRF

Одна из классических сильных архитектур до эпохи transformers:

```text
tokens → embeddings → BiLSTM → CRF
```

Почему нужен CRF сверху:

- softmax по каждому токену предсказывает метки независимо;
- CRF учитывает зависимости между соседними метками.

Например, после `B-PER` естественно ожидать `I-PER`, а не `B-LOC`.

## 10.4. Функция потерь

### Token-level softmax
$$
\mathcal{L} = -\sum_{t=1}^{T}\log P(y_t \mid h_t)
$$

### Для CRF
Используется нормализованная log-likelihood всей последовательности меток.

## 10.5. Применение

- POS tagging;
- named entity recognition;
- chunking;
- shallow parsing;
- biomedical tagging.

## 10.6. Ограничения

- последовательная обработка;
- сложнее параллелить;
- на очень длинных контекстах уступает transformers.

---

# 11. Seq2Seq Models

## 11.1. Интуитивная идея

Seq2Seq решает задачу:

**одна последовательность на входе → другая последовательность на выходе**.

Это особенно важно для:

- machine translation;
- summarization;
- dialogue response generation.

Курс вводит neural machine translation именно через encoder-decoder seq2seq и подчёркивает, что модель переводит последовательность переменной длины в другую последовательность переменной длины через фиксированное внутреннее представление.

## 11.2. Архитектура encoder-decoder

### Encoder
Читает вход:

$$
x_1,\dots,x_T
$$

и строит финальное скрытое состояние:

$$
h_T^{enc}
$$

### Decoder
Использует это состояние как начальную память и пошагово генерирует:

$$
y_1,\dots,y_{T'}
$$

Курс описывает именно такую схему: encoder состоит из embedding layer и LSTM; decoder тоже состоит из embedding layer и LSTM; decoder стартует с SOS token и затем использует предыдущее сгенерированное слово как вход следующего шага.

## 11.3. Формальная постановка

Вероятность выходной последовательности:

$$
P(y_{1:T'} \mid x_{1:T})
=
\prod_{t=1}^{T'} P(y_t \mid y_{<t}, c)
$$

где $c$ — контекстный вектор от encoder.

В базовом seq2seq:

$$
c = h_T^{enc}
$$

## 11.4. Teacher Forcing

Во время обучения decoder часто получает истинный предыдущий токен, а не своё собственное предсказание:

$$
P(y_t \mid y_{t-1}^{gold}, h_{t-1})
$$

Это ускоряет и стабилизирует обучение.

## 11.5. Потери

$$
\mathcal{L}
=
-\sum_{t=1}^{T'} \log P(y_t^{gold} \mid y_{<t}^{gold}, x)
$$

---

# 12. Главная проблема классического Seq2Seq: information bottleneck

## 12.1. Интуитивная идея

В базовом encoder-decoder вся информация входной последовательности должна быть сжата в один фиксированный вектор $c$. Это и есть **bottleneck**.

Курс прямо называет это major limitation of the traditional seq2seq model: fixed length memory делает длинные последовательности проблемными.

## 12.2. Почему это плохо

Если предложение длинное, то одному вектору трудно сохранить:

- точный синтаксис;
- дальние зависимости;
- локальные alignments между словами исходного и целевого языка.

Именно отсюда рождается attention, который будет в следующем разделе.

---

# 13. Decoding в sequence models

После обучения seq2seq нужно ещё уметь **генерировать** последовательность.

## 13.1. Greedy decoding

На каждом шаге выбрать самый вероятный токен:

$$
y_t = \arg\max_y P(y \mid y_{<t}, x)
$$

### Плюсы
- быстро;
- просто.

### Минусы
- локально оптимально, но не обязательно глобально;
- может пропустить лучшую целую последовательность.

## 13.2. Beam Search

## Интуитивная идея

Вместо одной гипотезы храним $k$ лучших частичных последовательностей. На каждом шаге расширяем каждую из них и снова оставляем лучшие $k$.

Это стандартный компромисс между greedy и полным перебором.

Хотя в субтитрах beam search подробно появляется уже в блоке neural machine translation рядом с attention, по учебной логике его удобно вводить здесь как естественный алгоритм декодирования sequence models.

## Формальная постановка

Хотим найти:

$$
\hat y = \arg\max_y P(y \mid x)
$$

Но пространство последовательностей слишком велико.

Beam search на шаге $t$:

1. держит набор гипотез $B_{t-1}$;
2. расширяет каждую всеми возможными токенами;
3. считает score;
4. оставляет top-$k$.

### Score
Обычно:

$$
\text{score}(y_{1:t}) = \sum_{i=1}^{t}\log P(y_i \mid y_{<i}, x)
$$

Используются логарифмы по той же причине, что и раньше: стабильность.

## Псевдокод

```text
beam = [([SOS], 0.0)]

for t in 1..max_len:
    candidates = []
    for seq, score in beam:
        probs = decoder_next_token_distribution(seq, x)
        for token in top_tokens(probs):
            new_seq = seq + [token]
            new_score = score + log(probs[token])
            candidates.append((new_seq, new_score))
    beam = top_k(candidates, k=beam_size)

return best_finished_sequence(beam)
```

## Ограничения
- beam search не гарантирует глобальный optimum;
- может порождать слишком частотные, «безопасные» ответы;
- beam size влияет на качество и скорость.

---

# 14. Применения sequence models в NLP

## 14.1. Sentiment Analysis
Many-to-one RNN:
$$
x_1,\dots,x_T \rightarrow y
$$

Курс использует sentiment analysis как один из типичных примеров, где RNN переносит информацию от начала фразы к концу для итогового решения.

## 14.2. Language Modeling
$$
P(x_{t+1}\mid x_{1:t})
$$

Используется для:
- autocomplete;
- text generation;
- scoring candidate sequences.

## 14.3. Machine Translation
Encoder-decoder:
$$
x_{1:T} \rightarrow y_{1:T'}
$$

Курс прямо строит neural machine translation на seq2seq with LSTMs.

## 14.4. Sequence Tagging
$$
x_1,\dots,x_T \rightarrow y_1,\dots,y_T
$$

POS, NER, chunking.

## 14.5. Caption Generation
One-to-many:
image $\rightarrow$ sequence of words.

Курс использует caption generation как пример другой RNN-архитектуры по типу входов/выходов.

---

# 15. Практический pipeline sequence NLP

```text
text
→ tokenization
→ embeddings
→ sequence encoder (RNN / LSTM / GRU / BiLSTM)
→ task-specific head
→ loss
→ BPTT
→ decoding / inference
```

### Для классификации
```text
tokens → embeddings → RNN/LSTM → final hidden state → classifier
```

### Для tagging
```text
tokens → embeddings → BiLSTM → softmax / CRF
```

### Для generation / translation
```text
source tokens → encoder → context → decoder → greedy / beam search
```

---

# 16. Короткие Python-примеры

## 16.1. Простая RNN-модель на PyTorch

```python
import torch
import torch.nn as nn

class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)              # [B, T, E]
        output, h_n = self.rnn(emb)          # h_n: [1, B, H]
        last_hidden = h_n[-1]                # [B, H]
        logits = self.fc(last_hidden)
        return logits
```

## 16.2. LSTM для классификации

```python
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        output, (h_n, c_n) = self.lstm(emb)
        last_hidden = h_n[-1]
        logits = self.fc(last_hidden)
        return logits
```

## 16.3. BiLSTM для token classification

```python
import torch
import torch.nn as nn

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, x):
        emb = self.embedding(x)              # [B, T, E]
        output, _ = self.lstm(emb)           # [B, T, 2H]
        logits = self.classifier(output)     # [B, T, num_tags]
        return logits
```

## 16.4. Encoder-decoder sketch

```python
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        emb = self.embedding(src)
        outputs, (h, c) = self.lstm(emb)
        return h, c

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h, c):
        emb = self.embedding(x)
        output, (h, c) = self.lstm(emb, (h, c))
        logits = self.fc(output)
        return logits, h, c
```

## 16.5. Beam search skeleton

```python
import math

def beam_search_step(beam, next_token_probs_fn, beam_size=3):
    candidates = []
    for seq, score in beam:
        probs = next_token_probs_fn(seq)  # dict[token] = prob
        for token, p in probs.items():
            candidates.append((seq + [token], score + math.log(p + 1e-12)))
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:beam_size]
```

---

# 17. Ограничения sequence models

Это важная часть. Sequence models были огромным прогрессом, но не финальной точкой.

## 17.1. Long-range dependencies
Даже LSTM/GRU не идеально удерживают очень длинный контекст.

## 17.2. Sequential computation
Шаг $t$ зависит от шага $t-1$, поэтому модель трудно параллелить.

## 17.3. Bottleneck в seq2seq
Один вектор контекста плохо описывает длинное предложение.

## 17.4. Alignment problem
При переводе полезно знать, на какие именно слова входа сейчас опираться. Базовый seq2seq этого не делает явно.

Именно из этих ограничений курс подводит к следующей теме — attention. В субтитрах это сформулировано очень прямо: после seq2seq разбираются its shortcomings and the solution that leads to the model used in subsequent assignments.

---

# 18. Что нужно усвоить по этому разделу

После этого раздела ты должен уметь:

1. объяснить, чем sequence models лучше n-gram models;
2. записать уравнение простой RNN;
3. объяснить, что такое hidden state;
4. понимать BPTT;
5. объяснить vanishing и exploding gradients;
6. описать, как LSTM решает проблему памяти;
7. различать LSTM и GRU;
8. понимать, зачем нужен bidirectional context;
9. различать many-to-one, many-to-many и encoder-decoder architectures;
10. объяснить seq2seq и information bottleneck;
11. описать greedy decoding и beam search;
12. понимать, почему sequence models привели к attention models.

---

# 19. Итог раздела

Sequence models — это этап, на котором NLP научился воспринимать текст как **динамический поток контекста**, а не просто как набор частот или коротких n-gram windows.

Именно здесь впервые в полном виде появляются:

- learned contextual states;
- recurrent memory;
- token-by-token generation;
- encoder-decoder mapping между последовательностями;
- neural sequence tagging.

Главный исторический смысл этого этапа:

### Было
- Bag of Words;
- Naive Bayes;
- n-grams;
- HMM.

### Стало
- RNN;
- LSTM;
- GRU;
- seq2seq;
- neural translation;
- contextual token representations.

Но sequence models всё ещё упираются в bottleneck, длинные зависимости и слабую параллелизацию.

Отсюда естественно вырастает следующий шаг:

**RNN / LSTM / seq2seq → attention → transformer**.